# Week 09: Tensors Store Model Data and Parameters

This notebook follows the reviewed Week 9 presentation. Read each concept and calculate the worked example before running its code.

## Lesson map

1. Tensors Store Model Data and Parameters
2. One Neuron Is a Weighted Sum and Activation
3. Layers Transform Batches with Matrix Multiplication
4. Activations Let Networks Learn Nonlinear Patterns
5. A Forward Pass Produces Predictions
6. The Loss Must Match the Task and Output
7. Backpropagation Computes Gradients
8. The Optimizer Applies the Gradients
9. Batches, Epochs, and Learning Rate Control Training
10. Training and Evaluation Modes Behave Differently
11. A Complete Training Loop Produces Evidence
12. Guided Lab: Trace One Batch End to End

Use the same reasoning loop throughout: **predict, run, inspect, explain**.


## 1. Tensors Store Model Data and Parameters

A PyTorch **tensor** is a shaped numerical data structure used for inputs, targets, model parameters, and intermediate results.

Like a NumPy array, a tensor has:

- `shape`: length of each axis;
- `dtype`: stored numerical type;
- `device`: CPU, GPU, or another accelerator;
- values selected by indexing.

PyTorch tensors can record operations for automatic gradient calculation.

### Work it out first

A batch contains `4` students and `3` features per student.

Input shape: `(4, 3)`  
If the model predicts one value per student, output shape: `(4, 1)`.

### Notebook bridge

The PyTorch notebook starts with tensor creation, operations, and device movement.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
import torch

x = torch.tensor([[2.0, 3.0], [4.0, 1.0]])
print(x.shape, x.dtype, x.device)

Expected output:

```text
torch.Size([2, 2]) torch.float32 cpu
```


## 2. One Neuron Is a Weighted Sum and Activation

A **neuron** calculates:

`z = w1x1 + w2x2 + ... + wnxn + b`

then applies an **activation function**:

`a = g(z)`

- `xi`: input feature
- `wi`: learned weight
- `b`: learned bias
- `z`: pre-activation weighted sum
- `g`: activation function
- `a`: neuron output

The bias has the same role as a regression intercept.

### Work it out first

`x=[2,3]`, `w=[0.4,-0.1]`, `b=0.2`

`z = 2(0.4) + 3(-0.1) + 0.2`  
`= 0.8 - 0.3 + 0.2 = 0.7`

If `g` is ReLU, output remains `0.7`.

### Notebook bridge

PyTorch `nn.Linear` performs the weighted sum and bias for a complete layer.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
x = torch.tensor([2.0, 3.0])
w = torch.tensor([0.4, -0.1])
z = x @ w + 0.2
print(z)

Expected output:

```text
tensor(0.7000)
```


## 3. Layers Transform Batches with Matrix Multiplication

A **layer** contains several neurons that receive the same input vector.

For a batch:

`Z = XWᵀ + b`

If:

- `X` shape is `(batch, input_features)`;
- `W` shape is `(output_features, input_features)`;
- then `Z` shape is `(batch, output_features)`.

Each output column comes from one neuron's weights.

### Work it out first

Batch `X`: `(4, 3)`  
Layer: `3` inputs to `5` outputs  
Weight `W`: `(5, 3)`  
Output `Z`: `(4, 5)`

Four observations each receive five transformed features.

### Notebook bridge

The model section constructs layers whose dimensions must match dataset tensors.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
layer = torch.nn.Linear(3, 5)
x = torch.randn(4, 3)
z = layer(x)
print(z.shape)

Expected output:

```text
torch.Size([4, 5])
```


## 4. Activations Let Networks Learn Nonlinear Patterns

An **activation function** transforms each pre-activation value.

ReLU:

`ReLU(z) = max(0, z)`

Without nonlinear activations, several linear layers collapse into one linear transformation. Nonlinearity lets the network represent curved and piecewise relationships.

Common output activations depend on the task. PyTorch loss functions often expect raw logits, so do not add sigmoid or softmax blindly.

### Work it out first

Input values `[-2, 0.5, 3]`

ReLU:

`max(0,-2)=0`  
`max(0,0.5)=0.5`  
`max(0,3)=3`

Output `[0, 0.5, 3]`.

### Notebook bridge

Learners should identify every activation and the shape it preserves.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
values = torch.tensor([-2.0, 0.5, 3.0])
print(torch.relu(values))

Expected output:

```text
tensor([0.0000, 0.5000, 3.0000])
```


## 5. A Forward Pass Produces Predictions

A **forward pass** sends input tensors through layers and activations to produce predictions.

Example network:

`X -> Linear(3,5) -> ReLU -> Linear(5,1) -> output`

The output of one stage becomes the input to the next. Shapes must align at every boundary.

The final output may be a regression prediction, a binary logit, or one logit per class.

### Work it out first

Batch shape `(4,3)`

After first linear layer: `(4,5)`  
After ReLU: `(4,5)`  
After output layer: `(4,1)`

There is one raw output per observation.

### Notebook bridge

The notebook's model call executes `forward()` and returns a tensor for the loss function.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
model = torch.nn.Sequential(
    torch.nn.Linear(3, 5),
    torch.nn.ReLU(),
    torch.nn.Linear(5, 1),
)
print(model(torch.randn(4, 3)).shape)

Expected output:

```text
torch.Size([4, 1])
```


## 6. The Loss Must Match the Task and Output

A **loss function** compares model outputs with targets and returns a value training will reduce.

- Regression: mean squared error can compare numerical predictions and targets.
- Binary classification: `BCEWithLogitsLoss` combines sigmoid and binary cross-entropy safely.
- Multiclass classification: `CrossEntropyLoss` expects one raw logit per class and integer class labels.

Using the wrong target shape, target type, or extra activation can produce errors or incorrect learning.

### Work it out first

Regression predictions `[2,4]`, targets `[3,5]`

Residuals `[1,1]`  
Squared residuals `[1,1]`  
MSE `(1+1)/2 = 1`

### Notebook bridge

Learners must explain why the notebook's chosen loss matches its prediction task.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
loss_fn = torch.nn.MSELoss()
prediction = torch.tensor([2.0, 4.0])
target = torch.tensor([3.0, 5.0])
print(loss_fn(prediction, target))

Expected output:

```text
tensor(1.)
```


## 7. Backpropagation Computes Gradients

PyTorch **autograd** records tensor operations in a computational graph.

During **backpropagation**, the chain rule moves backward from loss and calculates:

`∂loss / ∂parameter`

for every trainable parameter.

Calling `loss.backward()` computes gradients. It does not update the weights. Gradients are stored in each parameter's `.grad`.

### Work it out first

`prediction = wx`, target `y`, squared loss `(y - wx)^2`.

If `x=2`, `w=1`, `y=5`:

Prediction `2`; residual `3`; loss `9`.  
Derivative with respect to `w`: `-2x(y-wx) = -12`.

Increasing `w` should reduce loss.

### Notebook bridge

The training loop calls `backward()` after calculating loss.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
w = torch.tensor(1.0, requires_grad=True)
loss = (5 - w * 2) ** 2
loss.backward()
print(w.grad)

Expected output:

```text
tensor(-12.)
```


## 8. The Optimizer Applies the Gradients

For each training batch:

1. `optimizer.zero_grad()` clears old gradients;
2. `prediction = model(x)` runs the forward pass;
3. `loss.backward()` computes new gradients;
4. `optimizer.step()` updates parameters.

The **optimizer** implements the update rule. Stochastic gradient descent and Adam are common choices. The **learning rate** controls update size.

### Work it out first

Current weight `1.0`, gradient `-12`, learning rate `0.1`:

`w_new = 1.0 - 0.1(-12)`  
`= 2.2`

For the example on Slide 7, this moves the prediction from `2` toward target `5`.

### Notebook bridge

This sequence is the core of the notebook training section.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
optimizer.zero_grad()
prediction = model(x)
loss = loss_fn(prediction, y)
loss.backward()
optimizer.step()

Expected output:

```text
Parameters are updated once using gradients from the current batch.
```


## 9. Batches, Epochs, and Learning Rate Control Training

A **batch** is a subset of training observations used for one gradient update.

An **epoch** is one complete pass through the training dataset.

If there are `1,000` rows and batch size is `100`, one epoch contains `10` batches and usually `10` optimizer updates.

Smaller batches use less memory and produce noisier gradient estimates. Larger batches use more memory. More epochs do not guarantee better generalization.

### Work it out first

Dataset size `240`, batch size `32`.

Seven full batches contain `224` rows, plus one final batch of `16`.  
Updates per epoch: `ceil(240/32) = 8`.

For `5` epochs: `8 x 5 = 40` updates.

### Notebook bridge

Learners should calculate expected batches and explain logged loss frequency.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
import math

updates = math.ceil(240 / 32) * 5
print(updates)

Expected output:

```text
40
```


## 10. Training and Evaluation Modes Behave Differently

`model.train()` enables training-time behaviour.  
`model.eval()` enables evaluation-time behaviour for layers such as dropout and batch normalization.

During evaluation, use `torch.inference_mode()` or `torch.no_grad()` to avoid storing gradients.

The model and all input tensors must be on compatible devices. Moving only the model or only the data causes a device mismatch.

### Work it out first

Correct evaluation sequence:

1. `model.eval()`
2. enter `torch.inference_mode()`
3. move batch to the model device
4. calculate predictions and metrics
5. do not call optimizer methods

### Notebook bridge

The notebook evaluation section must use both correct model mode and no-gradient execution.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
model.eval()
with torch.inference_mode():
    predictions = model(x_valid)

Expected output:

```text
Validation predictions without gradient tracking or parameter updates.
```


## 11. A Complete Training Loop Produces Evidence

A training loop should expose:

- input and target shapes;
- forward output shape;
- loss value;
- gradient calculation;
- parameter update;
- training and validation metrics by epoch;
- random seed and device;
- saved best model based on validation evidence.

Training loss alone does not establish generalization. Validation must run without updates.

### Work it out first

If training loss falls from `1.2` to `0.1` while validation loss falls to `0.4` then rises to `0.9`, later epochs may be overfitting.

The preferred checkpoint is near the lowest validation loss, not automatically the final epoch.

### Notebook bridge

This maps directly to the real-world PyTorch notebook's model, training, and evaluation sections.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
for x, y in train_loader:
    optimizer.zero_grad()
    prediction = model(x)
    loss = loss_fn(prediction, y)
    loss.backward()
    optimizer.step()

Expected output:

```text
One parameter update per batch; epoch metrics are logged separately.
```


## 12. Guided Lab: Trace One Batch End to End

Build a small network and document:

1. input and target tensor shapes;
2. device and dtype;
3. every layer's input and output shape;
4. weighted sum and activation for one simple neuron;
5. matching task and loss;
6. one forward loss value;
7. one parameter gradient after `backward()`;
8. one parameter value before and after `step()`;
9. batch count and epoch count;
10. validation in evaluation and inference modes.

### Work it out first

For `240` rows, batch `32`, and `5` epochs:

`8` batches per epoch and `40` updates. A validation pass performs `0` updates.

### Notebook bridge

Complete `02.real-world-pytorch.ipynb`, predicting shapes and state changes before each section.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
before = next(model.parameters()).detach().clone()
loss.backward()
optimizer.step()
after = next(model.parameters()).detach().clone()
print(torch.equal(before, after))

Expected output:

```text
False
```


## Guided lab

Build a small network and document:

1. input and target tensor shapes;
2. device and dtype;
3. every layer's input and output shape;
4. weighted sum and activation for one simple neuron;
5. matching task and loss;
6. one forward loss value;
7. one parameter gradient after `backward()`;
8. one parameter value before and after `step()`;
9. batch count and epoch count;
10. validation in evaluation and inference modes.

### Reference result

For `240` rows, batch `32`, and `5` epochs:

`8` batches per epoch and `40` updates. A validation pass performs `0` updates.


In [ ]:
# Guided lab workspace: Week 09
# Add only the imports needed for the current step.

# TODO 1: Prepare the smallest valid input.

# TODO 2: Apply the concept taught in this lesson.

# TODO 3: Display inspectable intermediate evidence.

# TODO 4: Compare the result with a hand calculation or stated requirement.

## Weekly deliverable

Submit the completed guided lab with:

- your prediction before execution;
- intermediate values, shapes, metrics, or traces;
- one failed assumption and its correction;
- a plain-English explanation of the result;
- the source notebook section you are now ready to complete.


## Sources and source notebooks

- <https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html>
- <https://github.com/curiousily/AI-Bootcamp/blob/master/02.real-world-pytorch.ipynb>
- <https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html>
- <https://d2l.ai/chapter_multilayer-perceptrons/mlp.html>
- <https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html>
- <https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html>
- <https://d2l.ai/chapter_multilayer-perceptrons/mlp.html#activation-functions>
- <https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html>
- <https://docs.pytorch.org/docs/stable/nn.html#loss-functions>
- <https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html>
- <https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html>
- <https://docs.pytorch.org/tutorials/beginner/introyt/autogradyt_tutorial.html>
- <https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html>
- <https://docs.pytorch.org/docs/stable/optim.html>
- <https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html>
- <https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.eval>
- <https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html>
- <https://docs.pytorch.org/tutorials/beginner/basics/>